In [16]:
import pandas as pd
import fastparquet

In [17]:
# Arquivos de Populacao

pop10 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a10anos-RIPSA.xlsx')
pop12 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a12anos-RIPSA.xlsx')
pop11a59 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-11a59-RIPSA.xlsx')
pop60 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-60+RIPSA.xlsx')
pop_geral = pd.read_excel('Dados-apoio/proj-2015-2019-POP-GERAL-RIPSA.xlsx')

pops = [pop12,pop11a59,pop60,pop_geral]

In [18]:
# Carrega os bancos Basico (com distancias e tempos), municipio por CIR e RAS e Sinan
base01 = pd.read_excel('Dados-iniciais/base01.xlsx')
muni_cir = pd.read_excel('Dados-iniciais/_Muni_por_Macro_DRS_CIR.xlsx')
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Dados-processados/2_df_recodificado.parquet', 
                        filters=[('ANO', 'in', list(range(2015, 2020)))] # Com filtro interno
)

# Padroniza codigo municipio do Sinan
sinan['ID_MN_RESI'] = sinan['ID_MN_RESI'].astype(int)

##########################################################
# Cria variavel Moderado/Grave (MG)
dfp = sinan

# converter ampolas para número
for col in ["NU_AMPOL_8", "NU_AMPOL_9"]:
    dfp[col] = pd.to_numeric(dfp[col], errors="coerce").fillna(0)

# =========================================================
# FILTRO DE CASOS MODERADOS/GRAVES QUALIFICADOS
# ESCORPIONISMO
# =========================================================

# ---------------------------------------------------------
# CRITÉRIOS FORTES
# Isoladamente já sugerem fortemente MG
# ---------------------------------------------------------

criterio_forte = (

    # SAA >= 2 ampolas
    (dfp["NU_AMPOL_8"] >= 2) |

    # SAEsc >= 2 ampolas
    (dfp["NU_AMPOL_9"] >= 2) |

    # Óbito por animais peçonhentos
    (dfp["EVOLUCAO"] == "Obito por ap") |

    # Manifestações vagais
    (dfp["CLI_VAGAIS"] == "Sim")

)

# ---------------------------------------------------------
# CRITÉRIOS ASSOCIATIVOS
# Variáveis sujeitas a erro de preenchimento,
# mas que em conjunto aumentam a probabilidade
# de representar MG
# ---------------------------------------------------------

criterio_associativo = (

    # Soroterapia + classificação moderado/grave
    (
        (dfp["CON_SOROTE"] == "Sim") &
        (dfp["TRA_CLASSI"].isin(["Moderado", "Grave"]))
    ) |

    # Soroterapia + manifestações sistêmicas
    (
        (dfp["CON_SOROTE"] == "Sim") &
        (dfp["MCLI_SIST"] == "Sim")
    ) |

    # Soroterapia + complicações sistêmicas
    (
        (dfp["CON_SOROTE"] == "Sim") &
        (dfp["COM_SISTEM"] == "Sim")
    )

)

# ---------------------------------------------------------
# FILTRO FINAL
# ---------------------------------------------------------

filtro_mg = criterio_forte | criterio_associativo

# ---------------------------------------------------------
# APLICAR FILTRO
# ---------------------------------------------------------

dfp_mg = dfp[filtro_mg].copy()

# ---------------------------------------------------------
# CRIAR VARIÁVEL BINÁRIA (OPCIONAL)
# ---------------------------------------------------------

dfp["TOTAL_MG"] = filtro_mg.astype(object) # aceita texto e booleanos

# ---------------------------------------------------------
# CONFERÊNCIA
# ---------------------------------------------------------

print("Total de casos:", len(dfp))
print("Moderados/Graves qualificados:", filtro_mg.sum())
print("Proporção:", round(filtro_mg.mean() * 100, 2), "%")


# Recodificando valores
dfp.loc[dfp['TOTAL_MG'] == False, 'TOTAL_MG'] = 'Leve'
dfp.loc[dfp['TOTAL_MG'] == True, 'TOTAL_MG'] = 'MG'

Total de casos: 117357
Moderados/Graves qualificados: 4475
Proporção: 3.81 %


In [19]:
# Junta todos os arquivos de populacao

pop_merge = pop10.copy() # Cria uma cópia para não mexer no original

for i in pops:
    # O merge traz as colunas novas e você salva o resultado em pop_merge
    pop_merge = pop_merge.merge(
        right=i.iloc[:, [0, 2]], 
        how='left', 
        on='IBGE'
    )

In [20]:
# Junta o banco de populacoes com a base 01

df = (
    base01
    .drop_duplicates()
    .merge(
        pop_merge.drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

In [21]:
# Junta df com banco de regioes
df = (
    df.merge(
    right=muni_cir,
    how='left',
    left_on="MUNI_NOME",
    right_on="MUNI_NOME",
    indicator='merge_flag'
).copy()
)

In [22]:
# Cria colunas de totais de casos por faixa etaria
total_casos = sinan['ID_MN_RESI'].value_counts().reset_index(name='TOTAL_CASOS')


In [23]:
# Junta total de casos ao banco base
df = df.merge(
    right=total_casos,
    right_on='ID_MN_RESI',
    left_on='IBGE',
    how='left'
    )


In [24]:
# Junta TOTAL_MG ao banco base

df = df.merge(
    right=dfp['ID_MN_RESI'],
    right_on='ID_MN_RESI',
    left_on='IBGE',
    how='left'
    )

In [29]:
df

,ACESSO_LOCAL,MULTIPLO_PESA,REGIAO,PESA,MUNI_REFERENCIADO,OBSERVACOES,LAT_MUNI,LON_MUNI,LAT_PESA,LON_PESA,...,MACRO_CODIGO,MACRO_NOME,DRS_CODIGO,DRS_NOME,CIR_CODIGO,CIR_NOME,merge_flag,ID_MN_RESI_x,TOTAL_CASOS,ID_MN_RESI_y
0,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,-50.064911,...,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both,350110.0,89.0,350110.0
1,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,-50.064911,...,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both,350110.0,89.0,350110.0
2,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,-50.064911,...,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both,350110.0,89.0,350110.0
3,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,-50.064911,...,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both,350110.0,89.0,350110.0
4,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,-50.064911,...,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both,350110.0,89.0,350110.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117361,1,0,CARAGUATATUBA,ILHABELA,ILHABELA,TODOS,NaN,NaN,NaN,NaN,...,3526,RRAS17,3517,DRS-17 Taubate,35173,Litoral Norte,both,352040.0,8.0,352040.0
117362,1,0,CARAGUATATUBA,ILHABELA,ILHABELA,TODOS,NaN,NaN,NaN,NaN,...,3526,RRAS17,3517,DRS-17 Taubate,35173,Litoral Norte,both,352040.0,8.0,352040.0
117363,1,0,CARAGUATATUBA,ILHABELA,ILHABELA,TODOS,NaN,NaN,NaN,NaN,...,3526,RRAS17,3517,DRS-17 Taubate,35173,Litoral Norte,both,352040.0,8.0,352040.0
117364,1,0,CARAGUATATUBA,ILHABELA,ILHABELA,TODOS,NaN,NaN,NaN,NaN,...,3526,RRAS17,3517,DRS-17 Taubate,35173,Litoral Norte,both,352040.0,8.0,352040.0


In [25]:
dfp.head(2)

,DT_SIN_PRI,SEM_PRI,ANO_NASC,NU_IDADE_N,CS_SEXO,CS_GESTANT,CS_RACA,CS_ESCOL_N,ID_MN_RESI,ID_OCUPA_N,...,ANO,MES,CBO,OCUPACAO,NOME_MUNI,POPULACAO,MACRO_NOME,DRS_NOME,CIR_NOME,TOTAL_MG
0,2019-04-12,49,"1983,0",4035,Masculino,Nao se apl ica,Branca,5 a 8 incompleto,351040,914405,...,2019.0,4.0,914405,Mecânico de manutenção de automóveis e veículo...,CAPIVARI,51369.0,RRAS14,DRS-10 Piracicaba,Piracicaba,Leve
1,2019-02-11,44,"1982,0",4037,Feminino,Nao,Parda,NaN,350950,998999,...,2019.0,2.0,NaN,NaN,CAMPINAS,1187974.0,RRAS15,DRS-07 Campinas,Regiao Metropolitana de Campinas,Leve


In [28]:
df.to_excel('Dados-iniciais/base02.xlsx')

In [27]:
'''
# Codigo comparacao

comparacao = (
    base01[["MUNI_REFERENCIADO"]]
    .drop_duplicates()
    .merge(
        pop_merge[["MUNI_NOME"]].drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

nao_encontrados = comparacao[comparacao["_merge"] == "left_only"]

print(nao_encontrados)
'''

'\n# Codigo comparacao\n\ncomparacao = (\n    base01[["MUNI_REFERENCIADO"]]\n    .drop_duplicates()\n    .merge(\n        pop_merge[["MUNI_NOME"]].drop_duplicates(),\n        left_on="MUNI_REFERENCIADO",\n        right_on="MUNI_NOME",\n        how="left",\n        indicator=True\n    )\n)\n\nnao_encontrados = comparacao[comparacao["_merge"] == "left_only"]\n\nprint(nao_encontrados)\n'